# Demo Person Re-ID đa camera — 3 hướng, CHUNG 1 video cố định
**Cách chạy:** Runtime → Change runtime type → **T4 GPU** → chạy lần lượt từng ô từ trên xuống.
Video cố định: phố đi bộ Calle Preciados (Madrid), 19 giây — các script tự tải, không cần upload gì.
Chuỗi 4 việc để kể khi trình bày: **Thấy người (YOLO) → Bám theo bằng Local ID (BoT-SORT) → Nhận ra qua camera bằng mã ngoại hình (OSNet) → Gộp Global ID**.
Hướng 4 (DeepStream/MV3DT + travel-time/topology) không chạy — chỉ trình bày kiến trúc + số liệu paper.

In [ ]:
# === PHẦN 0: CÀI THƯ VIỆN (chạy 1 lần, khoảng 2-4 phút) ===
# Nếu Colab báo "Restart runtime" sau ô này thì bấm Restart rồi chạy tiếp từ Phần tải video.
!pip install -q torchreid ultralytics insightface onnxruntime-gpu "numpy==1.26.4"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CHƯA CÓ GPU — vào Runtime > Change runtime type > T4 GPU")

In [ ]:
# === TẢI VIDEO CỐ ĐỊNH (dùng chung cho cả 3 hướng) ===
# Phố đi bộ Calle Preciados (Madrid), 19 giây, 720p — Wikimedia Commons
!wget -q "https://upload.wikimedia.org/wikipedia/commons/0/0b/Calle_preciados.ogv" -O /content/video.ogv
!ffmpeg -y -loglevel error -i /content/video.ogv -q:v 3 /content/video.mp4
# Weights OSNet train trên MSMT17 (bản ImageNet mặc định cho điểm SAI — đừng dùng)
!wget -q "https://huggingface.co/kaiyangzhou/osnet/resolve/main/osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth" -O /content/weights_osnet_msmt17.pth
import os
print("Video:", os.path.getsize("/content/video.mp4")//1024, "KB | Weights:", os.path.getsize("/content/weights_osnet_msmt17.pth")//1024//1024, "MB")

## Hướng 1 — Re-ID bằng ngoại hình (Torchreid + OSNet)
Ý tưởng: mỗi người có một "mã ngoại hình" (dãy 512 số). Cùng người thì mã gần nhau dù chụp ở thời điểm/góc khác nhau.

In [ ]:
# === HƯỚNG 1: Re-ID BẰNG NGOẠI HÌNH ===
# Luồng: bám người bằng YOLO+BoT-SORT -> cắt crop TO NHẤT của cùng 1 ID + crop 1 người khác
#        -> OSNet trích "mã ngoại hình" -> ma trận độ giống. Kỳ vọng: cùng người ~0.86, khác người ~0.40
import cv2, numpy as np
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
samples = {}
results = model.track(source="/content/video.mp4", tracker="botsort.yaml", classes=[0], stream=True, verbose=False)
for fi, r in enumerate(results):
    if r.boxes is None or r.boxes.id is None:
        continue
    img = r.orig_img
    for box, tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.cpu().numpy().astype(int)):
        x1, y1, x2, y2 = [int(v) for v in box]
        if (x2 - x1) < 40 or (y2 - y1) < 90:
            continue
        lst = samples.setdefault(int(tid), [])
        if not lst or fi - lst[-1][0] >= 30:
            lst.append((fi, img[y1:y2, x1:x2].copy(), (x2 - x1) * (y2 - y1)))
ranked = sorted(samples.items(), key=lambda kv: len(kv[1]), reverse=True)
main_id, ml = ranked[0]
other_id, ol = ranked[1]
ml_by_area = sorted(ml, key=lambda t: -t[2])
ol_by_area = sorted(ol, key=lambda t: -t[2])
first = ml_by_area[0]
second = next((t for t in ml_by_area[1:] if abs(t[0] - first[0]) >= 45), ml_by_area[1])
jobs = [("crop_cungnguoi_1.jpg", first[1]), ("crop_cungnguoi_2.jpg", second[1]), ("crop_khacnguoi.jpg", ol_by_area[0][1])]
paths = []
for name, crop in jobs:
    p = f"/content/{name}"
    cv2.imwrite(p, crop)
    paths.append(p)
print(f"Cùng người = ID {main_id}; khác người = ID {other_id}. Đã cắt 3 crops.")

try:
    from torchreid.utils import FeatureExtractor
except ImportError:
    from torchreid.reid.utils import FeatureExtractor
extractor = FeatureExtractor(model_name="osnet_x1_0", model_path="/content/weights_osnet_msmt17.pth", device="cuda")
feats = extractor(paths)
feats = (feats / feats.norm(dim=1, keepdim=True)).cpu().numpy()
sims = feats @ feats.T
names = [os.path.basename(p) for p in paths]
print("\nMa trận độ giống OSNet (1.0 = giống hệt):")
print("               " + "".join(f"{n[:12]:>13}" for n in names))
for i, n in enumerate(names):
    print(f"{n[:12]:>12} " + "".join(f"{sims[i, j]:>13.3f}" for j in range(3)))
print("\nĐọc kết quả: crop_cungnguoi_1 <-> crop_cungnguoi_2 phải CAO hơn hẳn so với crop_khacnguoi.")

## Hướng 2 — Tracking + Local ID (YOLO + BoT-SORT)
Mỗi người trong video được gắn 1 số áo tạm (Local ID) và bám theo suốt clip. Đây là việc số 2 trong chuỗi 4 việc.

In [ ]:
# === HƯỚNG 2: TRACKING — VIDEO CÓ LOCAL ID ===
# Chạy tracking trên chính video cố định, lưu video kết quả để tải về.
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.track(source="/content/video.mp4", tracker="botsort.yaml", classes=[0], save=True, verbose=False)
print("Xong. Mở thư mục runs/detect/track ở panel Files bên trái, tải file video.mp4 về.")
print("Lưu ý trình bày: số trên khung là Local ID — chỉ có nghĩa trong 1 camera này.")

## Hướng 3 — Fusion đa tín hiệu + bảng ablation
Cộng điểm ngoại hình + khuôn mặt. Bảng 3 cột là bằng chứng để trả lời câu "vì sao 0.6/0.4?". Chạy SAU Hướng 1.

In [ ]:
# === HƯỚNG 3: FUSION + BẢNG SO SÁNH (ABLATION) ===
# Dùng lại 3 crops của Hướng 1: in 3 cột — chỉ ngoại hình / chỉ khuôn mặt / fusion 0.6-0.4
import numpy as np, cv2
from insightface.app import FaceAnalysis
try:
    from torchreid.utils import FeatureExtractor
except ImportError:
    from torchreid.reid.utils import FeatureExtractor

paths = ["/content/crop_cungnguoi_1.jpg", "/content/crop_cungnguoi_2.jpg", "/content/crop_khacnguoi.jpg"]
extractor = FeatureExtractor(model_name="osnet_x1_0", model_path="/content/weights_osnet_msmt17.pth", device="cuda")
feats = extractor(paths)
feats = (feats / feats.norm(dim=1, keepdim=True)).cpu().numpy()
body = {(0, 1): float(feats[0] @ feats[1]), (0, 2): float(feats[0] @ feats[2])}

app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
def face_emb(p):
    img = cv2.imread(p)
    img = cv2.resize(img, None, fx=2, fy=2)
    faces = app.get(img)
    return faces[0].normed_embedding if faces else None
embs = [face_emb(p) for p in paths]

print(f"{'Cặp':<26}{'Ngoại hình':>12}{'Khuôn mặt':>16}{'Fusion':>22}")
for label, i, j in [("cùng người (2 thời điểm)", 0, 1), ("khác người", 0, 2)]:
    b = body[(i, j)]
    f = float(embs[i] @ embs[j]) if embs[i] is not None and embs[j] is not None else None
    fused = 0.6 * b + 0.4 * f if f is not None else None
    print(f"{label:<26}{b:>12.3f}{(f'{f:.3f}' if f is not None else 'không thấy mặt'):>16}{(f'{fused:.3f}' if fused is not None else 'chỉ dùng ngoại hình'):>22}")
print("\nCâu nói với anh Tiến: trọng số 0.6/0.4 là heuristic chọn bằng thử trên tập cặp demo;")
print("không detect được mặt thì fallback về ngoại hình — hệ thống không sập.")